# MWE: sparse images with gradient CLEAN, starlet L1 and log-sum penalties

Maximum entropy and the other smoothness penalties prefer images whose flux is spread out. Some scenes are not like that: a few compact knots, perhaps with one extended feature, next to a bright star. This notebook reconstructs such a scene with the sparse tools of `virgil.imaging` (see `design/sparse_imaging.md`):
- **`clean`**, gradient CLEAN. It adds point components one at a time where they lower χ² most, using the gradient of the likelihood, so it works for DISCO and closure phases, not only for calibrated visibilities;
- **`StarletL1`**, an L1 norm of the image's wavelet (starlet) coefficients, which prefers images made of few compact structures at any scale;
- **`LogSum`**, a smooth surrogate for the number of bright pixels (SQUEEZE's L0).

An L1 norm of the pixels themselves would do nothing: an `Image`'s pixels are positive and sum to one, so their L1 norm is always one.

The data are simulated JWST AMI DISCOs of a star with three compact knots and one extended blob, 3.5% of the flux in all.

In [ ]:
import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()
)
sys.path.insert(0, str(root / "src"))

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpyro.distributions as dist

from virgil.coverage import ami_grid_record
from virgil.imaging import (
    LogSum,
    MaxEntropy,
    StarletL1,
    beam,
    clean,
    convolve_beam,
    field_of_view,
    image_priors,
    l_curve,
)
from virgil.likelihood import whitened_residuals
from virgil.models import Image, PointSource, System, circular_support
from virgil.oidata import OIData
from virgil.plotting import plot_model
from virgil.scenes import gaussian_blob

template = OIData(ami_grid_record(wavelength_m=4.8e-6, rotation_deg=-6.9))
scale = 20.0
npix = round(field_of_view(template, largest_mas=1250.0) / scale)
fov = npix * scale
features = [  # (dra, ddec, sigma, flux): three knots and a blob
    (-200.0, 140.0, 10.0, 0.010),
    (180.0, 80.0, 10.0, 0.006),
    (60.0, -220.0, 10.0, 0.007),
    (-120.0, -140.0, 60.0, 0.012),
]
truth_image = sum(
    f * gaussian_blob(npix, scale, s, dra=x, ddec=y) for x, y, s, f in features
)
flux = float(truth_image.sum())
truth = System(
    star=PointSource(),
    env=Image.from_brightness(truth_image, scale, flux=flux),
)
data = template.with_model(truth, key=jax.random.PRNGKey(0))
resolution = beam(data)
chi2_truth = jnp.mean(whitened_residuals(truth, data) ** 2)
print(
    f"{data.n_independent} data; {npix}² pixels of {scale:.0f} mas; beam "
    f"{resolution.major_mas:.0f} × {resolution.minor_mas:.0f} mas; "
    f"the truth has χ²/N = {chi2_truth:.3f}"
)

## Gradient CLEAN

`clean` starts from the star alone (`base=PointSource()`). Each iteration computes the gradient of χ² with respect to the flux of a point at every pixel, picks the pixel where a Gauss–Newton step would lower χ² most, and adds a tenth (the loop gain) of that step. It stops when χ² per point reaches one. For data linear in the image this is exactly Högbom's CLEAN. A hole of half a beam under the star keeps CLEAN from trading the star's flux against pixels right next to it.

The components are point-like by construction, so the extended blob becomes a cluster of them. The restored image, the components convolved with the beam, shows what the data resolve.

In [ ]:
support = circular_support(
    npix, scale, radius_mas=fov / 2, inner_radius_mas=0.5 * resolution.minor_mas
)
result = clean(
    data, npix, scale, base=PointSource(), support=support, max_iterations=3000
)
print(
    f"stopped ({result.stop}) after {len(result.chi2_red) - 1} iterations at "
    f"χ²/N = {result.chi2_red[-1]:.3f}; {int((result.components > 0).sum())} "
    f"components with flux {float(result.components.sum()):.4f} "
    f"(truth {flux:.4f})"
)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].semilogy(result.chi2_red)
axes[0].axhline(1.0, color="k", ls=":", lw=1)
axes[0].set(xlabel="iteration", ylabel="χ² per point", title="CLEAN")
plot_model(truth.env, fov, npix, ax=axes[1], title="truth")
plot_model(result.model.clean, fov, npix, ax=axes[2], title="CLEAN components")
plot_model(
    result.model.clean, fov, npix, ax=axes[3], title="CLEAN, restored",
    beam=resolution, convolve=True,
)
plt.tight_layout()
plt.show()

## Regularised fits, started from CLEAN

The restored CLEAN image makes a good starting image for regularised maximum likelihood. Each regulariser's weight is swept with `l_curve`, from strong to weak, and chosen by the discrepancy principle (χ² per point reaches one). Maximum entropy is included for comparison. `LogSum` is not convex, so its starting image matters more than the others'.

In [ ]:
start = System(
    star=PointSource(),
    env=Image.from_brightness(
        jnp.maximum(convolve_beam(result.components, scale, resolution), 0.0),
        scale,
        floor=1e-3,
        support=support,
        flux=float(result.components.sum()),
    ),
)
priors = image_priors(start) | {"env.flux": dist.Uniform(0.0, 1.0)}
settings = {
    "MEM": (MaxEntropy(1.0, path="env"), jnp.logspace(-1, 4, 11)),
    "StarletL1": (StarletL1(1.0, path="env"), jnp.logspace(-2, 4, 13)),
    "LogSum": (LogSum(1.0, path="env"), jnp.logspace(-4, 2, 13)),
}
curves = {
    name: l_curve(start, priors, data, regulariser, weights, max_steps=50_000)
    for name, (regulariser, weights) in settings.items()
}


def chosen(curve):
    # The fit nearest the discrepancy weight, or the weakest if none.
    w = curve.discrepancy()
    if w is None:
        return len(curve.weights) - 1
    return int(jnp.argmin(jnp.abs(jnp.log(curve.weights) - jnp.log(w))))


for name, curve in curves.items():
    k = chosen(curve)
    fits = curve.results
    print(
        f"{name}: w = {float(curve.weights[k]):.3g}, χ²/N = "
        f"{fits[k].info['chi2_red']:.3f}, flux {float(fits[k].model.env.flux):.4f}; "
        f"all converged: {all(r.info['converged'] for r in fits)}"
    )

## Comparison

The top row shows each image at its native resolution, the bottom row convolved with the beam. NCC is the normalised cross-correlation with the truth (1 is perfect), at native resolution and after convolving both with the beam. "Top 2%" is the fraction of the image's flux in its brightest 2% of pixels, a measure of how compact it is.

In [ ]:
def ncc(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float(jnp.sum(a * b) / jnp.sqrt(jnp.sum(a * a) * jnp.sum(b * b)))


def top(image, fraction=0.02):
    flat = jnp.sort(image.ravel())[::-1]
    return float(flat[: max(1, int(fraction * flat.size))].sum() / flat.sum())


images = {"truth": truth.env, "CLEAN": result.model.clean} | {
    name: curve.results[chosen(curve)].model.env
    for name, curve in curves.items()
}
reference = truth.env.render(npix, fov)
smooth_reference = convolve_beam(reference, scale, resolution)
fig, axes = plt.subplots(2, len(images), figsize=(4 * len(images), 8))
for column, (name, image) in enumerate(images.items()):
    rendered = image.render(npix, fov)
    smooth = convolve_beam(rendered, scale, resolution)
    print(
        f"{name:10s} NCC {ncc(rendered, reference):.2f} native, "
        f"{ncc(smooth, smooth_reference):.2f} convolved; "
        f"top 2% {top(rendered):.2f}"
    )
    plot_model(
        image, fov, npix, ax=axes[0, column],
        title=f"{name}: NCC {ncc(rendered, reference):.2f}",
    )
    plot_model(
        image, fov, npix, ax=axes[1, column], beam=resolution, convolve=True,
        title=f"convolved: NCC {ncc(smooth, smooth_reference):.2f}",
    )
plt.tight_layout()
plt.show()

## Summary

SUMMARY_PLACEHOLDER